<a href="https://colab.research.google.com/github/TottiPuc/Machine_learning/blob/master/Anova_Insurance_ML_Model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Objective: To develop a predictive model to classify people as Healthy (0) or Unhealthy (1) using health variables, habits, and categorical data.



## 1. Library installation and import

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from google.colab import files

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report,
    RocCurveDisplay, ConfusionMatrixDisplay
)
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.inspection import permutation_importance
import joblib

RANDOM_STATE = 42

## 2. Loading the dataset


In [ ]:
uploaded = files.upload()

csv_name = list(uploaded.keys())[0]
df = pd.read_csv(csv_name)

print("File uploaded:", csv_name)
print("dataset Dimensions:", df.shape)
df.head()

Saving Healthcare_Data_Preprocessed_FIXED.csv to Healthcare_Data_Preprocessed_FIXED.csv
File uploaded: Healthcare_Data_Preprocessed_FIXED.csv
dataset Dimensions: (10000, 23)


,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type_Vegan,Diet_Type_Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,37.0,26.0,111.0,198.0,99.0,72.0,5.5,1.7,1.6,4.4,...,1,2,1,0.0,1.0,False,True,True,False,False
1,37.0,24.0,121.0,199.0,103.0,75.0,4.5,1.9,2.6,5.6,...,1,2,1,2.0,2.0,False,False,True,False,False
2,81.0,27.0,NaN,NaN,100.0,74.0,12.1,2.9,2.6,1.8,...,2,0,0,1.0,0.0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,4.3,1.0,1.7,5.2,...,1,2,1,2.0,0.0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,16.0,5.0,3.4,1.4,...,2,0,2,0.0,2.0,False,True,False,True,False


## 3. Initial review of the dataset

In [ ]:
display(df.info())
display(df.describe(include="all").T)

print("\nMissing values ​​per column:")
display(df.isna().sum().sort_values(ascending=False))



<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 23 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Age                   10000 non-null  float64
 1   BMI                   10000 non-null  float64
 2   Blood_Pressure        9471 non-null   float64
 3   Cholesterol           9462 non-null   float64
 4   Glucose_Level         9486 non-null   float64
 5   Heart_Rate            10000 non-null  float64
 6   Sleep_Hours           10000 non-null  float64
 7   Exercise_Hours        10000 non-null  float64
 8   Water_Intake          10000 non-null  float64
 9   Stress_Level          10000 non-null  float64
 10  Target                10000 non-null  int64  
 11  Smoking               10000 non-null  int64  
 12  Alcohol               10000 non-null  int64  
 13  Diet                  10000 non-null  int64  
 14  MentalHealth          10000 non-null  int64  
 15  PhysicalActivity    

None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Age,10000.0,NaN,NaN,NaN,40.228,24.350238,0.0,25.0,37.0,49.0,100.0
BMI,10000.0,NaN,NaN,NaN,25.7115,1.944594,19.0,24.0,26.0,27.0,32.0
Blood_Pressure,9471.0,NaN,NaN,NaN,130.922395,27.807917,22.0,114.0,135.0,150.0,225.0
Cholesterol,9462.0,NaN,NaN,NaN,199.193934,2.080687,192.0,198.0,199.0,200.0,207.0
Glucose_Level,9486.0,NaN,NaN,NaN,100.154122,2.205152,93.0,99.0,100.0,102.0,107.0
Heart_Rate,10000.0,NaN,NaN,NaN,73.5314,1.724329,67.0,72.0,74.0,75.0,80.0
Sleep_Hours,10000.0,NaN,NaN,NaN,8.00578,4.205662,0.0,5.0,6.3,10.8,16.0
Exercise_Hours,10000.0,NaN,NaN,NaN,2.43173,1.32928,0.0,1.5,2.0,3.3,5.0
Water_Intake,10000.0,NaN,NaN,NaN,2.47749,0.634218,0.6,2.0,2.4,2.9,4.6
Stress_Level,10000.0,NaN,NaN,NaN,4.50882,1.817407,0.0,3.3,4.8,5.8,10.0



Missing values ​​per column:


,0
Cholesterol,538
MedicalHistory,535
Blood_Pressure,529
Allergies,521
Glucose_Level,514
BMI,0
Age,0
Sleep_Hours,0
Heart_Rate,0
Exercise_Hours,0


In [ ]:
print("\nDistribution of the target variable:")
display(df["Target"].value_counts(dropna=False))
display(df["Target"].value_counts(normalize=True, dropna=False))


Distribution of the target variable:


,count
Target,
0,5001
1,4999


,proportion
Target,
0,0.5001
1,0.4999


## 4. Basic cleaning and quality control

The `Age` column can contain negative values. These will be treated as input errors and converted into missing values ​​for later imputation within the pipeline. Column names are also normalized for security purposes.

In [ ]:
data = df.copy()

data.columns = data.columns.str.strip().str.replace(" ", "_")

assert "Target" in data.columns, "The Target column was not found in the dataset."

data["Target"] = pd.to_numeric(data["Target"], errors="coerce")

if "Age" in data.columns:
    data.loc[data["Age"] < 0, "Age"] = np.nan

data = data.dropna(subset=["Target"])
data["Target"] = data["Target"].astype(int)

print("Dimension after initial cleaning:", data.shape)
data.head()

Dimension after initial cleaning: (10000, 23)


,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,Diet,MentalHealth,PhysicalActivity,MedicalHistory,Allergies,Diet_Type_Vegan,Diet_Type_Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O
0,37.0,26.0,111.0,198.0,99.0,72.0,5.5,1.7,1.6,4.4,...,1,2,1,0.0,1.0,False,True,True,False,False
1,37.0,24.0,121.0,199.0,103.0,75.0,4.5,1.9,2.6,5.6,...,1,2,1,2.0,2.0,False,False,True,False,False
2,81.0,27.0,NaN,NaN,100.0,74.0,12.1,2.9,2.6,1.8,...,2,0,0,1.0,0.0,True,False,False,False,False
3,25.0,21.0,150.0,199.0,102.0,70.0,4.3,1.0,1.7,5.2,...,1,2,1,2.0,0.0,True,False,False,True,False
4,24.0,26.0,146.0,202.0,99.0,76.0,16.0,5.0,3.4,1.4,...,2,0,2,0.0,2.0,False,True,False,True,False


## 5. Separation of predictor variables and target variable

In [ ]:
X = data.drop(columns=["Target"])
y = data["Target"]

numeric_features = X.select_dtypes(include=["int64", "float64", "int32", "float32"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()

print("Numerical Variables:", numeric_features)
print("Categorical Variables:", categorical_features)

Numerical Variables: ['Age', 'BMI', 'Blood_Pressure', 'Cholesterol', 'Glucose_Level', 'Heart_Rate', 'Sleep_Hours', 'Exercise_Hours', 'Water_Intake', 'Stress_Level', 'Smoking', 'Alcohol', 'Diet', 'MentalHealth', 'PhysicalActivity', 'MedicalHistory', 'Allergies']
Categorical Variables: ['Diet_Type_Vegan', 'Diet_Type_Vegetarian', 'Blood_Group_AB', 'Blood_Group_B', 'Blood_Group_O']


## 6. Train / Test Division



In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=RANDOM_STATE,
    stratify=y
)

print("Train:", X_train.shape, "Test:", X_test.shape)
print("Distribución y_train:")
print(y_train.value_counts(normalize=True))
print("Distribución y_test:")
print(y_test.value_counts(normalize=True))

Train: (8000, 22) Test: (2000, 22)
Distribución y_train:
Target
0    0.500125
1    0.499875
Name: proportion, dtype: float64
Distribución y_test:
Target
0    0.5
1    0.5
Name: proportion, dtype: float64


## 7. Preprocessing

Preprocessing is defined within a pipeline to prevent data leakage:

- Numerical variables: imputation using the median + scaling.

- Categorical variables: imputation using the mode + one-hot encoding.

In [ ]:
from sklearn.preprocessing import FunctionTransformer

def to_object_dataframe(X_input):
    X_df = pd.DataFrame(X_input).copy()
    return X_df.astype("object")

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("to_object", FunctionTransformer(to_object_dataframe, validate=False)),
    ("imputer", SimpleImputer(strategy="constant", fill_value="missing")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ],
    remainder="drop"
)

## 8. Training

Three models are tested:

1. Logistic regression as an interpretable baseline.

2. Random Forest as a robust nonlinear model.

3. Gradient Boosting as an ensemble model.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE),
    "Random Forest": RandomForestClassifier(
        n_estimators=300,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(random_state=RANDOM_STATE)
}

scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "roc_auc": "roc_auc"
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results = []

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])

    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        scoring=scoring,
        cv=cv,
        n_jobs=-1
    )

    results.append({
        "model": name,
        "accuracy_mean": scores["test_accuracy"].mean(),
        "precision_mean": scores["test_precision"].mean(),
        "recall_mean": scores["test_recall"].mean(),
        "f1_mean": scores["test_f1"].mean(),
        "roc_auc_mean": scores["test_roc_auc"].mean()
    })

results_df = pd.DataFrame(results).sort_values(by="f1_mean", ascending=False)
display(results_df)

,model,accuracy_mean,precision_mean,recall_mean,f1_mean,roc_auc_mean
1,Random Forest,0.863625,0.856046,0.874471,0.865080,0.938962
2,Gradient Boosting,0.861250,0.859152,0.864215,0.861636,0.938392
0,Logistic Regression,0.786000,0.792065,0.775939,0.783823,0.869700


## 9. Selecting and adjusting the best model
The cost of classifying a truly unhealthy person as healthy can be high.

Therefore, in addition to F1 and ROC-AUC, it is worth paying special attention to the **recall of class 1: Unhealthy**.

In [ ]:
best_model_name = results_df.iloc[0]["model"]
print("Best base model according to F1:", best_model_name)

final_model = RandomForestClassifier(
    random_state=RANDOM_STATE,
    class_weight="balanced",
    n_jobs=-1
)

final_pipe = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", final_model)
])

param_grid = {
    "model__n_estimators": [200, 400],
    "model__max_depth": [None, 5, 10, 20],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    final_pipe,
    param_grid=param_grid,
    scoring="f1",
    cv=cv,
    n_jobs=-1,
    verbose=1
)

grid.fit(X_train, y_train)

print("Better parameters:")
print(grid.best_params_)
print("Better F1 Cross Validation:", grid.best_score_)

Best base model according to F1: Random Forest
Fitting 5 folds for each of 32 candidates, totalling 160 fits
Better parameters:
{'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 400}
Better F1 Cross Validation: 0.866820449110223


## 10. Final Assessment in Test

In [ ]:
best_pipe = grid.best_estimator_

y_pred = best_pipe.predict(X_test)
y_proba = best_pipe.predict_proba(X_test)[:, 1]

metrics = {
    "Accuracy": accuracy_score(y_test, y_pred),
    "Precision": precision_score(y_test, y_pred),
    "Recall": recall_score(y_test, y_pred),
    "F1-score": f1_score(y_test, y_pred),
    "ROC-AUC": roc_auc_score(y_test, y_proba)
}

metrics_df = pd.DataFrame(metrics, index=["Test"]).T
display(metrics_df)

print("\nClassification report:")
print(classification_report(y_test, y_pred, target_names=["Healthy", "Unhealthy"]))

,Test
Accuracy,0.872000
Precision,0.857692
Recall,0.892000
F1-score,0.874510
ROC-AUC,0.942317



Classification report:
              precision    recall  f1-score   support

     Healthy       0.89      0.85      0.87      1000
   Unhealthy       0.86      0.89      0.87      1000

    accuracy                           0.87      2000
   macro avg       0.87      0.87      0.87      2000
weighted avg       0.87      0.87      0.87      2000



## 11. Save Model

In [ ]:
model_path = "anova_health_classifier.joblib"
joblib.dump(best_pipe, model_path)

print(f"Save Model: {model_path}")
files.download(model_path)

Save Model: anova_health_classifier.joblib


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 12. Inference function for new candidates

In [ ]:
def predict_health_status(new_data: pd.DataFrame, threshold: float = 0.5):
    probabilities = best_pipe.predict_proba(new_data)[:, 1]
    predictions = (probabilities >= threshold).astype(int)

    result = new_data.copy()
    result["probability_unhealthy"] = probabilities
    result["predicted_class"] = predictions
    result["predicted_label"] = result["predicted_class"].map({
        0: "Healthy",
        1: "Unhealthy"
    })
    return result

# Example: predict the first 5 test results
example_predictions = predict_health_status(X_test.head(5), threshold=0.5)
display(example_predictions)

,Age,BMI,Blood_Pressure,Cholesterol,Glucose_Level,Heart_Rate,Sleep_Hours,Exercise_Hours,Water_Intake,Stress_Level,...,MedicalHistory,Allergies,Diet_Type_Vegan,Diet_Type_Vegetarian,Blood_Group_AB,Blood_Group_B,Blood_Group_O,probability_unhealthy,predicted_class,predicted_label
5949,28.0,24.0,139.0,201.0,106.0,75.0,6.4,1.6,1.7,5.4,...,2.0,0.0,False,True,True,False,False,0.2100,0,Healthy
1723,37.0,25.0,127.0,195.0,100.0,71.0,6.7,1.2,1.9,5.6,...,2.0,0.0,False,True,True,False,False,0.8200,1,Unhealthy
3372,2.0,24.0,120.0,198.0,103.0,73.0,0.9,1.0,1.8,7.1,...,0.0,2.0,False,False,False,True,False,0.2475,0,Healthy
9033,18.0,28.0,90.0,197.0,98.0,71.0,16.0,5.0,3.7,3.0,...,1.0,1.0,False,True,False,False,False,0.9650,1,Unhealthy
7017,37.0,21.0,165.0,196.0,100.0,74.0,5.7,2.1,2.6,5.2,...,1.0,2.0,False,False,False,False,False,0.0450,0,Healthy
